In [1]:
!pip install -q -U transformers accelerate requests

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import re
import json
import glob
import time
import zlib
import subprocess
import requests
import numpy as np
import pandas as pd
import torch
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from scipy.stats import spearmanr
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 33.5 MB/s eta 0:00:00
device: cuda


In [2]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
    attn_implementation="sdpa",
)
model.eval()
for p_ in model.parameters():
    p_.requires_grad_(False)

lm = model.model.language_model
lm_head = model.lm_head
layers = lm.layers
n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = hidden_dim // n_head
IMAGE_TOKEN_ID = getattr(model.config, "image_token_id", None) or model.config.image_token_index
assert len(layers) == n_layer
assert model.config.text_config._attn_implementation == "sdpa"
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim, image token id", IMAGE_TOKEN_ID)

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

32 language model layers, 32 heads, 128 head dim, image token id 32000


In [3]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "nocturnalnerd18"
SLUG = f"{USERNAME}/patch-llava-exact"
img_dir = "/kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_CACHE_DIR = "/kaggle/temp/vg"
os.makedirs(VG_CACHE_DIR, exist_ok=True)
VG_URLS = ["https://cs.stanford.edu/people/rak248/VG_100K", "https://cs.stanford.edu/people/rak248/VG_100K_2"]

STAGE = "A"
N_A_UNITS = 150
K_B = 40
MAX_HOURS = 9.0
PILOT_UNITS = 3
BATCH_CELLS = 12
BATCH_SAVE_SIZE = 50
UPLOAD_EVERY = 2
CHECK_TOL = 0.25
YES_ID, NO_ID = 3869, 1939
TAIL_N = 4
GROUPS = ["visual", "text", "tail"]
OUT_DIR = "/kaggle/working/patch_llava"
os.makedirs(OUT_DIR, exist_ok=True)
CHECK_CELL = {"kind": "rp", "layer": 0, "head": -1, "group": "visual"}


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


def find_vg_image_path(image_id):
    local = os.path.join(VG_CACHE_DIR, f"{image_id}.jpg")
    if os.path.exists(local):
        return local
    for attempt in range(3):
        for base in VG_URLS:
            try:
                r = requests.get(f"{base}/{image_id}.jpg", timeout=30)
            except requests.RequestException:
                continue
            if r.status_code == 200 and len(r.content) > 1000:
                with open(local + ".part", "wb") as f:
                    f.write(r.content)
                os.replace(local + ".part", local)
                return local
        time.sleep(2 * (attempt + 1))
    return None


def resolve_image_path(q):
    qid = q["question_id"]
    if qid.startswith("pope_"):
        return os.path.join(img_dir, f"{q['image_id']}.jpg")
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    if qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}
assert [processor.tokenizer.decode([i]).strip() for i in (YES_ID, NO_ID)] == ["Yes", "No"]


def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))


atp = {}
atp_files = sorted(glob.glob(f"{INPUT}/atp-llava-noising/**/*_batch_*.pt", recursive=True), key=lambda p: (os.path.basename(p).split("_batch_")[0], batch_number(p)))
n_failed = 0
for path in atp_files:
    for e in torch.load(path):
        if e["ok"]:
            atp[e["question_id"]] = {k: e[k] for k in (
                "question_id", "image_id", "hallucination_type", "stratum", "group", "source", "split", "sign", "seed",
                "delta_clean", "delta_corrupt", "TE", "noise_used", "head", "mlp",
            )}
        else:
            n_failed += 1
print(f"atp-llava-noising: {len(atp_files)} batch files, {len(atp)} ok entries, {n_failed} failed entries")
if not atp_files:
    print("datasets visible under", INPUT, ":", sorted(os.listdir(INPUT)))
assert len(atp) >= 0.98 * 8403, f"only {len(atp)} ok atp entries from {len(atp_files)} files; attach atp-llava-noising (all batches) under {INPUT}"

meta = pd.DataFrame([{k: v for k, v in a.items() if k not in ("head", "mlp")} for a in atp.values()])
meta["gap_clean"] = meta.sign * meta.delta_clean
fy_all = meta[meta.stratum == "false_yes"]
ty_all = meta[meta.stratum == "true_yes"]
rng = np.random.default_rng(0)
units = []
for (t, s, sp), g in fy_all.groupby(["hallucination_type", "source", "split"]):
    pool = ty_all[(ty_all.hallucination_type == t) & (ty_all.source == s) & (ty_all.split == sp)].set_index("question_id").gap_clean
    ids = g.question_id.tolist()
    gaps = g.set_index("question_id").gap_clean
    order = rng.permutation(len(ids))
    for rank, i in enumerate(order):
        qid = ids[i]
        ctrl = None
        if len(pool):
            ctrl = (pool - gaps[qid]).abs().idxmin()
            pool = pool.drop(ctrl)
        units.append({"fy": qid, "ctrl": ctrl, "key": (rank + 0.5) / len(ids), "stratum": f"{t}/{s}/{sp}"})
units.sort(key=lambda u: (u["key"], u["stratum"]))
print(len(units), "false_yes units,", sum(u["ctrl"] is not None for u in units), "with a matched true_yes control")
print(pd.crosstab(fy_all.hallucination_type, [fy_all.source, fy_all.split]))

stage_units = units[:N_A_UNITS] if STAGE == "A" else units[N_A_UNITS:]
stage_ids = [q for u in stage_units for q in (u["fy"], u["ctrl"]) if q is not None]
print(f"stage {STAGE}: {len(stage_units)} units, {len(stage_ids)} questions")
print(pd.Series([u["stratum"] for u in stage_units]).value_counts().to_string())

with open(f"{INPUT}/atp-llava-analysis/shortlist.json") as f:
    shortlist = json.load(f)


def cell_of(index, group):
    if index < n_layer * n_head:
        return {"kind": "head", "layer": index // n_head, "head": index % n_head, "group": group}
    return {"kind": "mlp", "layer": index - n_layer * n_head, "head": -1, "group": group}


def cell_name(c):
    base = f"L{c['layer']} H{c['head']}" if c["kind"] == "head" else f"L{c['layer']} MLP"
    return f"{base} {c['group']}"


def build_full_cells():
    out = []
    for r in shortlist["ranked"]:
        groups = {"all"} | {k.split("/")[-1] for k in r["evidence"] if k.split("/")[-1] in GROUPS}
        for g in sorted(groups):
            out.append(cell_of(r["index"], g))
    for r in shortlist["always_include"]:
        out.append(cell_of(r["index"], r["group"]))
    seen, uniq = set(), []
    for c in out:
        key = cell_name(c)
        if key not in seen:
            seen.add(key)
            uniq.append(c)
    return uniq


if STAGE == "A":
    cells_run = build_full_cells()
else:
    ok_, out_ = False, ""
    r_ = subprocess.run(["kaggle", "datasets", "download", "-d", SLUG, "-p", OUT_DIR, "--unzip"], capture_output=True, text=True)
    assert os.path.exists(f"{OUT_DIR}/cells_B.json"), "run stage A to the end first so cells_B.json is published"
    with open(f"{OUT_DIR}/cells_B.json") as f:
        cells_run = json.load(f)
print(len(cells_run), "patch cells per direction:", {k: sum(c["kind"] == k for c in cells_run) for k in ("head", "mlp")},
      "| groups", {g: sum(c["group"] == g for c in cells_run) for g in ["all"] + GROUPS})


def prep(qid):
    q = question_by_id[qid]
    inputs = build_inputs(load_image(resolve_image_path(q)), q["question_text"])
    ids = inputs["input_ids"]
    T = ids.shape[1]
    vis = ids[0] == IMAGE_TOKEN_ID
    tail = torch.zeros(T, dtype=torch.bool, device=ids.device)
    tail[-TAIL_N:] = True
    text = ~vis & ~tail
    assert int(vis.sum()) == 576 and not bool((vis & tail).any())
    return {
        "qid": qid, "T": T, "input_ids": ids, "pixel_values": inputs["pixel_values"],
        "attention_mask": inputs["attention_mask"], "vis": vis,
        "idx": {
            "all": torch.arange(T), "visual": torch.nonzero(vis.cpu())[:, 0],
            "text": torch.nonzero(text.cpu())[:, 0], "tail": torch.nonzero(tail.cpu())[:, 0],
        },
        "sign": 1.0 if q["ground_truth_answer"].strip().lower() == "yes" else -1.0,
    }

atp-llava-noising: 43 batch files, 8403 ok entries, 0 failed entries
3358 false_yes units, 2846 with a matched true_yes control
source             amber            pope           reefknot           
split               test train  val test train val     test train  val
hallucination_type                                                    
attribute            262  1207  267    0     0   0        0     0    0
object                 0     0    0   27   135  32        0     0    0
relation              14   107   24    0     0   0      192   901  190
stage A: 150 units, 300 questions
attribute/amber/train      54
relation/reefknot/train    40
attribute/amber/val        12
attribute/amber/test       12
relation/reefknot/test      9
relation/reefknot/val       8
object/pope/train           6
relation/amber/train        5
object/pope/val             1
object/pope/test            1
relation/amber/val          1
relation/amber/test         1
157 patch cells per direction: {'head': 74, 'mlp': 8

In [4]:
STATE = {"noise": None, "seed": 0, "capture": False, "live": None, "vis": None, "patch": None}


def _layer_pre(l):
    def hook(module, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        changed = False
        if l == 0 and STATE["noise"] is not None:
            vis = STATE["vis"]
            h = h.clone()
            x = h[:, vis].float()
            generator = torch.Generator().manual_seed(STATE["seed"])
            noise = torch.randn(x.shape, generator=generator).to(h.device)
            h[:, vis] = (x + noise * x.std() * STATE["noise"]).to(h.dtype)
            changed = True
        if STATE["capture"]:
            STATE["live"]["rp"][l] = h
        if not changed:
            return None
        if args:
            return (h,) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs)
        kwargs["hidden_states"] = h
        return args, kwargs
    return hook


def _z_pre(l):
    def hook(module, args):
        z = args[0]
        patch = STATE["patch"]
        if patch is not None and patch["layer"] == l and patch["heads"]:
            z = z.clone()
            B, T = z.shape[0], z.shape[1]
            z4 = z.view(B, T, n_head, head_dim)
            src = patch["src"]["z"][l][0].view(T, n_head, head_dim)
            for b, h_, idx in patch["heads"]:
                ii = idx.to(z.device)
                z4[b, ii, h_] = src[ii.to(src.device), h_].to(z.device, z.dtype)
            return (z,) + tuple(args[1:])
        if STATE["capture"]:
            STATE["live"]["z"][l] = z
        return None
    return hook


def _mlp_fwd(l):
    def hook(module, args, output):
        patch = STATE["patch"]
        if patch is not None and patch["layer"] == l and patch["mlps"]:
            out = output.clone()
            src = patch["src"]["mlp"][l][0]
            for b, idx in patch["mlps"]:
                ii = idx.to(out.device)
                out[b, ii] = src[ii.to(src.device)].to(out.device, out.dtype)
            return out
        if STATE["capture"]:
            STATE["live"]["mlp"][l] = output
        return None
    return hook


_handles = []
for l, layer in enumerate(layers):
    _handles.append(layer.register_forward_pre_hook(_layer_pre(l), with_kwargs=True))
    _handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_z_pre(l)))
    _handles.append(layer.mlp.register_forward_hook(_mlp_fwd(l)))
print(len(_handles), "hooks registered")

96 hooks registered


In [5]:
@torch.no_grad()
def capture_forward(p, noise, seed):
    STATE.update(noise=noise, seed=seed, capture=True, vis=p["vis"], patch=None)
    STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
    out = model(
        input_ids=p["input_ids"], pixel_values=p["pixel_values"], attention_mask=p["attention_mask"], use_cache=False, logits_to_keep=1,
    )
    logits = out.logits[0, -1].float()
    delta = p["sign"] * (logits[YES_ID] - logits[NO_ID])
    result = {"acts": STATE["live"], "delta": float(delta)}
    STATE.update(capture=False, live=None, noise=None)
    del out, logits
    return result


@torch.no_grad()
def patched_deltas(p, base, src, chunk):
    B = len(chunk)
    l0 = chunk[0]["layer"]
    h = base["acts"]["rp"][l0].expand(B, -1, -1).clone()
    heads, mlps = [], []
    for b, c in enumerate(chunk):
        assert c["layer"] == l0
        idx = p["idx"][c["group"]]
        if c["kind"] == "head":
            heads.append((b, c["head"], idx))
        elif c["kind"] == "mlp":
            mlps.append((b, idx))
        else:
            ii = idx.to(h.device)
            h[b, ii] = src["acts"]["rp"][l0][0, ii.to(src["acts"]["rp"][l0].device)].to(h.device, h.dtype)
    STATE.update(capture=False, noise=None, patch={"layer": l0, "heads": heads, "mlps": mlps, "src": src["acts"]})
    pos_dev = lm.rotary_emb.inv_freq.device
    pos = torch.arange(p["T"], device=pos_dev)[None]
    pe = lm.rotary_emb(h.to(pos_dev), pos)
    for l in range(l0, n_layer):
        out = layers[l](h, attention_mask=None, position_ids=pos, position_embeddings=pe)
        h = out[0] if isinstance(out, tuple) else out
    STATE["patch"] = None
    logits = lm_head(lm.norm(h[:, -1])).float()
    return (p["sign"] * (logits[:, YES_ID] - logits[:, NO_ID])).cpu().numpy()


def run_cells(p, base, src, cell_list):
    vals = np.zeros(len(cell_list))
    by_layer = {}
    for i, c in enumerate(cell_list):
        by_layer.setdefault(c["layer"], []).append(i)
    for l in sorted(by_layer):
        idxs = by_layer[l]
        for k in range(0, len(idxs), BATCH_CELLS):
            part = idxs[k:k + BATCH_CELLS]
            vals[part] = patched_deltas(p, base, src, [cell_list[i] for i in part])
    return vals


def exact_question(p, seed, noise, cell_list):
    clean = capture_forward(p, None, seed)
    corrupt = capture_forward(p, noise, seed)
    run = cell_list + [CHECK_CELL]
    v_noise = run_cells(p, clean, corrupt, run)
    v_denoise = run_cells(p, corrupt, clean, run)
    result = {
        "delta_clean": clean["delta"], "delta_corrupt": corrupt["delta"],
        "noising": torch.from_numpy((v_noise[:-1] - clean["delta"]).astype(np.float32)),
        "denoising": torch.from_numpy((v_denoise[:-1] - corrupt["delta"]).astype(np.float32)),
        "check_noising": float(v_noise[-1] - corrupt["delta"]), "check_denoising": float(v_denoise[-1] - clean["delta"]),
    }
    del clean, corrupt
    return result


def exact_with_retry(p, seed, noise, cell_list):
    global BATCH_CELLS
    for attempt in range(2):
        try:
            return exact_question(p, seed, noise, cell_list), None
        except torch.cuda.OutOfMemoryError:
            STATE.update(capture=False, live=None, noise=None, patch=None)
            torch.cuda.empty_cache()
            BATCH_CELLS = max(2, BATCH_CELLS // 2)
    return None, "oom"


def atp_value(a, c):
    x = a["head"][:, c["layer"], c["head"]] if c["kind"] == "head" else a["mlp"][:, c["layer"]]
    return float(x.sum() if c["group"] == "all" else x[GROUPS.index(c["group"])])

In [6]:
pilot_ids = [q for u in stage_units[:PILOT_UNITS] for q in (u["fy"], u["ctrl"]) if q is not None]
rows, secs, checks, base_err = [], [], [], []
for qid in pilot_ids:
    a = atp[qid]
    p = prep(qid)
    t0 = time.time()
    res, reason = exact_with_retry(p, a["seed"], a["noise_used"], cells_run)
    assert res is not None, reason
    secs.append(time.time() - t0)
    checks.append(max(abs(res["check_noising"]), abs(res["check_denoising"])))
    base_err.append(max(abs(res["delta_clean"] - a["delta_clean"]), abs(res["delta_corrupt"] - a["delta_corrupt"])))
    for i, c in enumerate(cells_run):
        rows.append({"qid": qid, "cell": cell_name(c), "kind": c["kind"], "atp": atp_value(a, c),
                     "noising": float(res["noising"][i]), "denoising": float(res["denoising"][i])})
    print(f"{qid} {a['stratum']:9s} T={p['T']}: {secs[-1]:.1f} s | clean {res['delta_clean']:+.2f} (atp run {a['delta_clean']:+.2f}) "
          f"corrupt {res['delta_corrupt']:+.2f} ({a['delta_corrupt']:+.2f}) | identity check noising {res['check_noising']:+.3f} denoising {res['check_denoising']:+.3f}")
    del p, res
    torch.cuda.empty_cache()

pil = pd.DataFrame(rows)
print(f"clean and corrupt deltas vs the attribution patching run: max abs diff {max(base_err):.3f}")
print(f"identity checks (patching the whole noise back in or out must reproduce the other run): max abs error {max(checks):.3f}")
print(f"pilot speed: {np.mean(secs):.1f} s per question for {len(cells_run)} cells in both directions")
print(f"projected time for stage {STAGE}: {np.mean(secs) * len(stage_ids) / 3600:.1f} h for {len(stage_ids)} questions")
print(f"attribution patching vs exact noising over {len(pil)} cells: spearman {spearmanr(pil.atp, pil.noising)[0]:.3f}, "
      f"slope of exact on atp {(pil.atp * pil.noising).sum() / (pil.atp ** 2).sum():.3f}")
for kind, sub in pil.groupby("kind"):
    print(f"  {kind:5s} spearman {spearmanr(sub.atp, sub.noising)[0]:.3f} over {len(sub)} cells")
print(f"noising vs denoising effects: spearman {spearmanr(pil.noising, pil.denoising)[0]:.3f}")
big = pil.reindex(pil.noising.abs().sort_values(ascending=False).index).head(12)
print(big[["qid", "cell", "atp", "noising", "denoising"]].round(3).to_string(index=False))
assert max(checks) < CHECK_TOL, "the batched patched forward does not reproduce the full forward"
assert max(base_err) < CHECK_TOL, "clean or corrupt deltas differ from the attribution patching collection"

amber_8434 false_yes T=597: 82.1 s | clean -2.14 (atp run -2.14) corrupt -1.10 (-1.10) | identity check noising +0.000 denoising +0.000
amber_2537 true_yes  T=596: 94.2 s | clean +2.14 (atp run +2.14) corrupt +0.82 (+0.82) | identity check noising +0.000 denoising +0.000
reefknot_2360672_1 false_yes T=603: 95.7 s | clean -0.97 (atp run -0.97) corrupt -0.36 (-0.36) | identity check noising +0.000 denoising +0.000
reefknot_2379411_0 true_yes  T=604: 96.5 s | clean +0.97 (atp run +0.97) corrupt +1.05 (+1.05) | identity check noising +0.000 denoising +0.000
amber_4846 false_yes T=598: 95.8 s | clean -1.73 (atp run -1.73) corrupt -0.09 (-0.09) | identity check noising +0.000 denoising +0.000
amber_6319 true_yes  T=596: 94.7 s | clean +1.73 (atp run +1.73) corrupt +0.66 (+0.66) | identity check noising +0.000 denoising +0.000
clean and corrupt deltas vs the attribution patching run: max abs diff 0.000
identity checks (patching the whole noise back in or out must reproduce the other run): max

In [7]:
def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    return r.returncode == 0 and "error" not in out.lower(), out


ok, out = _run(["kaggle", "datasets", "download", "-d", SLUG, "-p", OUT_DIR, "--unzip"])
if ok:
    print("restored", len(glob.glob(os.path.join(OUT_DIR, "*_batch_*.pt"))), "batch files from", SLUG)
else:
    if not any(s in out.lower() for s in ("404", "403", "not found", "does not exist", "forbidden")):
        raise RuntimeError("download failed for an unrelated reason, refusing to start fresh:\n" + out)
    print("no existing dataset, starting fresh")
dataset_exists = ok

with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "patch-llava-exact",
        "id": SLUG,
        "subtitle": "Exact activation patching in both directions for LLaVA yes/no questions",
        "description": "Exact activation patching on LLaVA-1.5-7B for every false_yes question of atp-llava-selection plus a true_yes control matched on the clean gap "
                       "within type, source and split. Corrupt run: Gaussian noise on the 576 visual embeddings at layer 0 (noise_used times their std, seed from atp-llava-noising). "
                       "noising [n_cells] is the change in delta when one component is replaced by its corrupt activation in the clean run (necessity), "
                       "denoising [n_cells] the change when it is replaced by its clean activation in the corrupt run (sufficiency). "
                       "delta is the sign-corrected Yes minus No logit gap at the last position. Cells are listed in cells_A.json and cells_B.json (kind head, mlp; layer; head; group all, visual, text, tail). "
                       "Stage A patches the full shortlist on the first units, stage B only the cells selected from stage A. check_noising and check_denoising are identity checks that must be near zero.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)


def upload(msg):
    global dataset_exists
    version_cmd = ["kaggle", "datasets", "version", "-p", OUT_DIR, "-m", msg]
    create_cmd = ["kaggle", "datasets", "create", "-p", OUT_DIR]
    ok_, out_ = _run(version_cmd if dataset_exists else create_cmd)
    if not ok_ and not dataset_exists:
        ok_, out2 = _run(version_cmd)
        out_ += "\n" + out2
    if ok_:
        dataset_exists = True
        print(f"    [upload ok] {msg}")
    else:
        print(f"    [UPLOAD FAILED, continuing] {out_[-400:]}")
    return ok_


with open(os.path.join(OUT_DIR, f"cells_{STAGE}.json"), "w") as f:
    json.dump(cells_run, f)


def collect():
    paths = sorted(glob.glob(os.path.join(OUT_DIR, f"{STAGE}_batch_*.pt")), key=batch_number)
    done_ids = set()
    for path in paths:
        done_ids.update(e["question_id"] for e in torch.load(path))
    todo = [q for q in stage_ids if q not in done_ids]
    print(f"stage {STAGE}: {len(stage_ids)} questions, {len(done_ids)} already done, {len(todo)} to run")

    reef = sorted({question_by_id[q]["image_id"] for q in todo if q.startswith("reefknot_")})
    if reef:
        with ThreadPoolExecutor(max_workers=16) as pool:
            got = list(pool.map(find_vg_image_path, reef))
        print(f"{len(reef) - sum(g is None for g in got)} of {len(reef)} Visual Genome images ready")

    batch_entries, batch_index, failed, missing = [], len(paths), 0, 0
    t_start = time.time()
    stopped = False
    for i, qid in enumerate(todo):
        q = question_by_id[qid]
        image_path = resolve_image_path(q)
        if image_path is None or not os.path.exists(image_path):
            missing += 1
            continue
        a = atp[qid]
        p = prep(qid)
        res, reason = exact_with_retry(p, a["seed"], a["noise_used"], cells_run)
        entry = {
            "question_id": qid, "image_id": q["image_id"], "hallucination_type": q["hallucination_type"],
            "stratum": a["stratum"], "group": a["group"], "source": a["source"], "split": a["split"], "stage": STAGE,
            "sign": p["sign"], "seed": a["seed"], "noise_used": a["noise_used"], "T": p["T"], "ok": res is not None,
            "atp_delta_clean": a["delta_clean"], "atp_delta_corrupt": a["delta_corrupt"],
        }
        if res is None:
            entry["reason"] = reason
            failed += 1
        else:
            entry.update(res)
        batch_entries.append(entry)
        del p

        if len(batch_entries) >= BATCH_SAVE_SIZE:
            torch.save(batch_entries, os.path.join(OUT_DIR, f"{STAGE}_batch_{batch_index}.pt"))
            batch_entries, batch_index = [], batch_index + 1
            if batch_index % UPLOAD_EVERY == 0:
                upload(f"stage {STAGE}: {batch_index} batches")
        if i % 25 == 0 and i > 0:
            torch.cuda.empty_cache()
            avg = (time.time() - t_start) / (i + 1)
            print(f"[{i + 1}/{len(todo)}] {qid} ({avg:.1f} s/question avg, ETA {avg * (len(todo) - i - 1) / 3600:.1f} h, failed {failed}, missing images {missing})")
        if time.time() - t_start > MAX_HOURS * 3600:
            stopped = True
            print(f"time budget of {MAX_HOURS} h reached after {i + 1} questions; re-run this notebook to resume")
            break

    if batch_entries:
        torch.save(batch_entries, os.path.join(OUT_DIR, f"{STAGE}_batch_{batch_index}.pt"))
    upload(f"stage {STAGE}: {'paused' if stopped else 'done'}")
    print(f"done -- stage {STAGE}, failed {failed}, missing images {missing}, {(time.time() - t_start) / 3600:.2f} h, complete {not stopped}")
    return missing, stopped


missing, stopped = collect()
assert missing == 0, f"{missing} questions have no image"

no existing dataset, starting fresh
stage A: 300 questions, 0 already done, 300 to run
112 of 112 Visual Genome images ready
[26/300] amber_5351 (95.6 s/question avg, ETA 7.3 h, failed 0, missing images 0)
[51/300] amber_5184 (95.6 s/question avg, ETA 6.6 h, failed 0, missing images 0)
[76/300] reefknot_2362946_0 (95.6 s/question avg, ETA 5.9 h, failed 0, missing images 0)
    [upload ok] stage A: 2 batches
[101/300] amber_1356 (95.7 s/question avg, ETA 5.3 h, failed 0, missing images 0)
[126/300] pope_540 (95.7 s/question avg, ETA 4.6 h, failed 0, missing images 0)
[151/300] reefknot_2326552_1 (95.6 s/question avg, ETA 4.0 h, failed 0, missing images 0)
[176/300] pope_2520 (95.6 s/question avg, ETA 3.3 h, failed 0, missing images 0)
    [upload ok] stage A: 4 batches
[201/300] reefknot_2408155_1 (95.7 s/question avg, ETA 2.6 h, failed 0, missing images 0)
[226/300] amber_4643 (95.6 s/question avg, ETA 2.0 h, failed 0, missing images 0)
[251/300] reefknot_2338608_1 (95.7 s/question avg

In [8]:
entries = []
for path in sorted(glob.glob(os.path.join(OUT_DIR, f"{STAGE}_batch_*.pt")), key=batch_number):
    entries.extend(torch.load(path))
ids = [e["question_id"] for e in entries]
print(f"{len(entries)} entries, {len(set(ids))} unique, {len(set(stage_ids) - set(ids))} stage questions not done yet")
assert len(ids) == len(set(ids)), "duplicated entries"
good = [e for e in entries if e["ok"]]
print(f"ok {len(good)}, failed {len(entries) - len(good)}")
print(pd.crosstab([pd.Series([e["hallucination_type"] for e in good], name="type"), pd.Series([e["source"] for e in good], name="source")],
                  pd.Series([e["stratum"] for e in good], name="stratum")))

check = max(max(abs(e["check_noising"]), abs(e["check_denoising"])) for e in good)
base = max(max(abs(e["delta_clean"] - e["atp_delta_clean"]), abs(e["delta_corrupt"] - e["atp_delta_corrupt"])) for e in good)
print(f"identity checks: max abs error {check:.3f} | clean and corrupt deltas vs the attribution patching run: max abs diff {base:.3f}")
bad_checks = [e["question_id"] for e in good if max(abs(e["check_noising"]), abs(e["check_denoising"])) > CHECK_TOL]
print(f"{len(bad_checks)} entries above the tolerance {CHECK_TOL}")
assert len(bad_checks) <= 0.02 * len(good), "too many entries fail the identity check"

N = np.stack([e["noising"].numpy() for e in good])
Dn = np.stack([e["denoising"].numpy() for e in good])
sg = np.array([e["sign"] for e in good])
A_pred = np.array([[atp_value(atp[e["question_id"]], c) for c in cells_run] for e in good])
stratum = np.array([e["stratum"] for e in good])
print(f"attribution patching vs exact noising over {N.size} entry-cells: spearman {spearmanr(A_pred.ravel(), N.ravel())[0]:.3f}, "
      f"slope {(A_pred * N).sum() / (A_pred ** 2).sum():.3f}")
mean_abs = np.abs(N).mean(0)
print(f"noising vs denoising, per-cell mean effect: spearman {spearmanr(N.mean(0), Dn.mean(0))[0]:.3f}")

if STAGE == "A":
    S_n = -sg[:, None] * N
    S_d = sg[:, None] * Dn
    fy_m, ct_m = stratum == "false_yes", stratum == "true_yes"
    assert fy_m.sum() >= 30 and ct_m.sum() >= 30, "too few stage A questions to choose stage B cells"

    def cohen(S):
        a, b = S[fy_m], S[ct_m]
        sp = np.sqrt(((len(a) - 1) * a.var(0, ddof=1) + (len(b) - 1) * b.var(0, ddof=1)) / (len(a) + len(b) - 2))
        return (a.mean(0) - b.mean(0)) / np.maximum(sp, 1e-6)

    d_score = np.maximum(np.abs(cohen(S_n)), np.abs(cohen(S_d)))
    mag_score = np.maximum(np.abs(S_n[fy_m].mean(0)), np.abs(S_d[fy_m].mean(0)))
    chosen = []
    for score in (d_score, mag_score):
        for i in np.argsort(-score):
            if int(i) not in chosen:
                chosen.append(int(i))
            if len(chosen) >= (K_B // 2 if score is d_score else K_B):
                break
    chosen = sorted(chosen)
    table = pd.DataFrame({
        "cell": [cell_name(cells_run[i]) for i in chosen], "d_score": d_score[chosen].round(2), "fy_mean_support_noising": S_n[fy_m].mean(0)[chosen].round(3),
        "fy_mean_support_denoising": S_d[fy_m].mean(0)[chosen].round(3),
    })
    print(table.sort_values("d_score", ascending=False).to_string(index=False))
    with open(os.path.join(OUT_DIR, "cells_B.json"), "w") as f:
        json.dump([cells_run[i] for i in chosen], f)
    print(len(chosen), "cells written to cells_B.json for stage B")

with open(os.path.join(OUT_DIR, f"patch_meta_{STAGE}.json"), "w") as f:
    json.dump({
        "model": MODEL_NAME, "yes_id": YES_ID, "no_id": NO_ID, "n_layer": n_layer, "n_head": n_head, "groups": GROUPS, "tail_n": TAIL_N,
        "stage": STAGE, "n_cells": len(cells_run), "n_ok": len(good), "n_failed": len(entries) - len(good), "max_check_error": check,
        "complete": len(set(stage_ids) - set(ids)) == 0,
    }, f)
assert upload(f"stage {STAGE}: verified"), "final upload failed"

300 entries, 300 unique, 0 stage questions not done yet
ok 300, failed 0
stratum             false_yes  true_yes
type      source                       
attribute amber            78        78
object    pope              8         8
relation  amber             7         7
          reefknot         57        57
identity checks: max abs error 0.016 | clean and corrupt deltas vs the attribution patching run: max abs diff 0.016
0 entries above the tolerance 0.25
attribution patching vs exact noising over 47100 entry-cells: spearman 0.803, slope 0.810
noising vs denoising, per-cell mean effect: spearman -0.092
          cell  d_score  fy_mean_support_noising  fy_mean_support_denoising
 L9 H12 visual     0.55                    0.039                     -0.006
     L9 H0 all     0.50                    0.067                     -0.183
    L9 H30 all     0.49                   -0.087                     -0.070
     L9 H5 all     0.48                    0.013                     -0.020
    L9